We are conducting a meta-analysis on vancomycin in total knee arthroplasty to reduce infection rates. There are 4 groups we are looking at:
 
No Vanco
Intravenous vancomycin
Topical vancomycin powder
Intraosseous vancomycin 
 
I have attached a spread sheet (in the format that you like- I hope) the outcomes are PJI at 90days, AKI at 30days, Vanco levels at 2hrs, SSI at 90d, VTE at 90 days and reoperation rate.
 
I was wondering if you could perform comparative stats on these for odds ratios, funnel and forest plots. 
 
Is it possible to perform these stats across all 4 groups at once? Alternatively, a subgroup analysis could be done for each outcome (i.e. nothing vs. topical, then IO vs IV, etc).
 
 
Thank you! Please let me know if there are any questions!

In [1]:
import pandas as pd
import re

input_file = "data/Vanco Stats.xlsx"
output_file = "data/Vanco Stats_formatted.xlsx"

# Mapping of treatment groups
groups = {
    'IV': ('n_events_IV_Vanco', 'n_total_IV'),
    'IO': ('n_events_IO', 'n_total_IO'),
    'None': ('n_events_none', 'n_total_none'),
    'Topical': ('n_events_topical', 'n_topcial_total')
}

def format_studyid(study):
    """
    Convert:
        Harper 2020  -> Harper (2020)
        Park 2025    -> Park (2025)
    """
    if pd.isna(study):
        return study

    study = str(study).strip()

    m = re.match(r'^(.*?)\s+(\d{4})$', study)
    if m:
        author, year = m.groups()
        return f"{author} ({year})"

    return study

# Read all sheets
xls = pd.ExcelFile(input_file)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet in xls.sheet_names:
        if sheet=='Serum Vanco Levels 2hr':
            continue

        df = pd.read_excel(xls, sheet_name=sheet)

        # Skip sheets without Study ID
        if 'Study ID' not in df.columns:
            continue

        long_dfs = []

        for group, (event_col, total_col) in groups.items():

            # Only process if columns exist in this sheet
            if event_col in df.columns and total_col in df.columns:

                temp = df[['Study ID', event_col, total_col]].copy()

                temp.columns = ['StudyID', 'n_events', 'n_total']
                temp['Group'] = group

                long_dfs.append(temp)

        if len(long_dfs) == 0:
            continue

        # Combine groups
        df_long = pd.concat(long_dfs, ignore_index=True)

        # Remove empty rows
        df_long = df_long.dropna(subset=['n_events', 'n_total'], how='all')

        # Format StudyID
        df_long['StudyID'] = df_long['StudyID'].apply(format_studyid)

        # Reorder columns
        df_long = df_long[['StudyID', 'Group', 'n_events', 'n_total']]
        df_long.dropna(inplace=True)

        # Save transformed sheet
        df_long.to_excel(writer, sheet_name=sheet, index=False)

print(f"Saved to {output_file}")

Saved to data/Vanco Stats_formatted.xlsx
